# Data Cleaning and Exploration — Adult Income Dataset

**Dataset:** Adult Income (UCI Machine Learning Repository)  
This dataset contains census data from 1994 with about 48,000 rows and 15 columns. It has info on age, education, occupation and whether someone earns above or below $50K a year.

Run all cells from top to bottom.

## Step 1 - Environment & Setup

In [ ]:
# uncomment to install if needed
# !pip install pandas numpy matplotlib seaborn

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns

print("Python:", sys.version)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("matplotlib:", matplotlib.__version__)
print("seaborn:", sns.__version__)

## Step 2 - Loading and Inspecting the Data

In [ ]:
# loading the dataset - the original UCI file uses ? for missing values
df = pd.read_csv("data/adult.csv", na_values="?")

print("shape:", df.shape)

In [ ]:
df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
# check data types
print(df.dtypes)

In [ ]:
# missing values
print(df.isnull().sum())

In [ ]:
# duplicates
print("duplicate rows:", df.duplicated().sum())

The Adult Income dataset has around 48,000 rows of US census data from 1994. Each row represents one person and includes 15 columns covering their demographics and employment details. Columns include things like age, education level, marital status, occupation and hours worked per week. The target column is income which tells us if the person earns more or less than $50K per year. Three columns have missing values - workclass, occupation and native_country - these were originally marked as ? in the raw data.

## Step 3 - Cleaning & Transformation

In [ ]:
# handle missing values
# these are all categorical so filling with the mode makes more sense than dropping rows
for col in ["workclass", "occupation", "native_country"]:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)
    print(f"filled {col} with: {mode_val}")

print("missing left:", df.isnull().sum().sum())

In [ ]:
# strip whitespace - the raw data has spaces before some values
str_cols = df.select_dtypes(include="str").columns
df[str_cols] = df[str_cols].apply(lambda c: c.str.strip())

print(df["income"].unique())

In [ ]:
# convert to proper types
df["income"] = df["income"].astype("category")
df["education"] = df["education"].astype("category")
df["sex"] = df["sex"].astype("category")

print(df[["income", "education", "sex"]].dtypes)

In [ ]:
# add a derived column for age group
def age_group(age):
    if age < 30:
        return "Young"
    elif age < 50:
        return "Middle"
    else:
        return "Senior"

df["age_group"] = df["age"].apply(age_group)
print(df["age_group"].value_counts())

In [ ]:
# export cleaned data
df.to_csv("clean_data.csv", index=False)
print("saved clean_data.csv")

## Step 4 - Exploration

In [ ]:
# filter to people working over 40 hours a week
overtime = df[df["hours_per_week"] > 40].copy()
print("rows:", len(overtime))

In [ ]:
# table 1 - average age and hours by income group
# people who earn more tend to be older and work longer hours
table1 = df.groupby("income")[["age", "hours_per_week"]].mean().round(2)
print(table1)

In [ ]:
# table 2 - count by education and income
# higher education levels tend to have more people in the >50K group
table2 = df.groupby(["education", "income"]).size().unstack(fill_value=0)
print(table2)

In [ ]:
# table 3 - average hours worked per occupation
# some occupations clearly have much longer average working hours
table3 = df.groupby("occupation")["hours_per_week"].mean().round(2).sort_values(ascending=False)
print(table3)

## Step 5 - Custom Function

In [ ]:
# function that takes a dataframe and column name and returns basic stats
# includes count, min, max, mean, median and range
def column_summary(dataframe, column_name):
    col = dataframe[column_name].dropna()
    
    result = {
        "count": int(col.count()),
        "min": float(col.min()),
        "max": float(col.max()),
        "mean": round(float(col.mean()), 2),
        "median": float(col.median()),
        "range": float(col.max() - col.min())
    }
    return result

In [ ]:
# testing on 3 columns
print("age:", column_summary(df, "age"))
print()
print("hours_per_week:", column_summary(df, "hours_per_week"))
print()
print("capital_gain:", column_summary(df, "capital_gain"))